# ML Feature Importance - What Predicts Cognitive Decline?

The idea here is simple: take a bunch of health and lifestyle data from patients at their first visit, feed it into a machine learning model, and see which factors best predict whether someone is cognitively normal or impaired.

Then, instead of just trusting the model blindly, SHAP values break open the black box and show exactly how each factor pushed the prediction one way or the other.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import shap

%matplotlib inline
np.random.seed(42)

## Loading the data

The full dataset is huge, so only the first 50k rows get loaded here. That is more than enough for exploration.

In [ ]:
file_path = '../data/investigator_nacc72.csv'
print("Loading data...")
try:
    df = pd.read_csv(file_path, nrows=50000, low_memory=False)
    print(f"Shape: {df.shape}")
except FileNotFoundError:
    print(f"Could not find {file_path}")
    df = pd.DataFrame()

## Cleaning and prep

Patients show up multiple times (one row per visit), so this grabs just the first visit for each person. That way the model looks at baseline health, not repeated measurements.

The features below cover demographics, cardiovascular health, mental health, substance use, neurological history, and baseline cognitive score.

In [ ]:
if not df.empty:
    # Keep only the first visit per patient
    df_baseline = df.sort_values(by=['NACCID', 'VISITYR']).drop_duplicates(subset=['NACCID'], keep='first')
    print(f"Unique patients: {df_baseline.shape[0]}")

    target = 'NACCUDSD'  # 1=Normal, 2=Impaired not MCI, 3=MCI, 4=Dementia

    features = [
        # Demographics
        'NACCAGE', 'SEX', 'EDUC',
        # Cardiovascular
        'HYPERTEN', 'DIABETES', 'HYPERCHO', 'NACCBMI',
        'CVHATT', 'CVAFIB', 'CVANGIO', 'CVBYPASS',
        'CVPACDEF', 'CVPACE', 'CVCHF', 'CVANGINA', 'CVOTHR',
        # Mental health and sleep
        'DEP', 'SLEEPAP',
        # Substance use
        'TOBAC30', 'ALCOHOL',
        # Neurological / trauma
        'PD', 'PDOTHR', 'SEIZURES', 'TBI', 'TBIOTHR',
        # Cognitive baseline
        'NACCMMSE'
    ]

    existing_features = [f for f in features if f in df_baseline.columns]

    if target in df_baseline.columns:
        model_data = df_baseline[existing_features + [target]].copy()
        model_data = model_data[model_data[target].isin([1, 2, 3, 4])]

        # Negative values in this dataset mean missing/unknown, replace with median
        model_data[existing_features] = model_data[existing_features].mask(model_data[existing_features] < 0, np.nan)
        model_data = model_data.fillna(model_data.median())

        print(f"Clean data shape: {model_data.shape}")
    else:
        print("Target variable NACCUDSD not found.")

## Training the model

This trains a Random Forest to classify patients as either **Normal** or **Impaired** (anything above Normal). The `class_weight='balanced'` flag makes sure the model does not just default to predicting Normal every time, since there are more Normal patients in the dataset.

In [ ]:
if 'model_data' in locals() and not model_data.empty:
    y = (model_data[target] > 1).astype(int)  # 0 = Normal, 1 = Impaired
    X = model_data[existing_features]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, y_pred))
    print()
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Impaired']))

## SHAP - what actually matters?

SHAP breaks open the model and shows, for each patient, how every single feature nudged the prediction toward Normal or Impaired.

**How to read the chart below:**
- Each row is a feature, ranked top to bottom by overall importance.
- Each dot is one patient.
- **Color** = the actual value of that feature for that patient. Red means high, blue means low.
- **Horizontal position** = how that feature affected the prediction. Right of center means it pushed toward Impaired. Left means it pushed toward Normal.

So for example, if red dots for `NACCAGE` (high age) cluster on the right, that means older age increases the predicted risk of impairment.

In [ ]:
if 'rf' in locals():
    explainer = shap.TreeExplainer(rf)

    X_test_sample = X_test.sample(n=min(1000, len(X_test)), random_state=42)
    shap_values = explainer.shap_values(X_test_sample)

    if isinstance(shap_values, list):
        shap_to_plot = shap_values[1]  # class 1 = Impaired
    elif len(shap_values.shape) == 3:
        shap_to_plot = shap_values[:, :, 1]
    else:
        shap_to_plot = shap_values

    shap.summary_plot(shap_to_plot, X_test_sample, feature_names=existing_features)